# AI-Powered Coding Tutor — RAG Embedding Pipeline on Kaggle Cloud GPU

**Mục tiêu:** Thực thi toàn bộ quá trình Vector Embedding cho kho tri thức Coding Tutor trên hạ tầng Kaggle GPU (T4 x2 hoặc P100), hoàn toàn **không sử dụng tài nguyên máy cá nhân**.

### Kiến trúc Định tuyến Modality (Modality-routed RAG Architecture):
- **Dữ liệu đầu vào:** `all_chunks.json` (5,230 chunks)
- **Định tuyến Modality (Dominance-based Rule):**
  - `TEXT` (**4,287 chunks** - 82.0%): Lý thuyết, giải thích, bài tập, phân tích lỗi, bảng biểu, công thức KaTeX $\rightarrow$ **`Qwen/Qwen3-Embedding-0.6B`** (1024-dim, Context 32K).
  - `CODE` (**943 chunks** - 18.0%): Mã nguồn mẫu, hàm, lớp, thuật toán lập trình $\rightarrow$ **`jinaai/jina-embeddings-v2-base-code`** (768-dim, Context 8K).
- **Cơ chế an toàn:**
  - Resilient Checkpointing: Tự động lưu theo batch, Resume thông minh khi notebook bị ngắt.
  - Sequential Loading: Nạp tuần tự từng model, giải phóng VRAM bằng `torch.cuda.empty_cache()` để tránh OOM.
  - Không lưu trữ bất kỳ token hay credentials nào.

In [ ]:
# =============================================================================
# 1. KIỂM TRA PHẦN CỨNG GPU & MÔI TRƯỜNG KAGGLE
# =============================================================================
import os
import sys
import subprocess

print("=== [1/7] KIỂM TRA PHẦN CỨNG GPU TRÊN KAGGLE ===")
try:
    gpu_info = subprocess.check_output(["nvidia-smi"], text=True)
    print(gpu_info)
except Exception as e:
    print(f"⚠️ Không tìm thấy lệnh nvidia-smi hoặc đang chạy CPU: {e}")

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Count: {torch.cuda.device_count()}")
    print(f"Active GPU Name:  {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:       {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("❌ CẢNH BÁO: Chưa bật GPU Accelerator! Vui lòng vào Notebook Settings -> Accelerator -> GPU T4 x2 hoặc P100.")

In [ ]:
# =============================================================================
# 2. CÀI ĐẶT DEPENDENCIES TRONG MÔI TRƯỜNG KAGGLE
# =============================================================================
!pip install -q --upgrade transformers sentence-transformers accelerate pyarrow fastparquet

In [ ]:
# =============================================================================
# 3. THIẾT LẬP ĐƯỜNG DẪN INPUT & OUTPUT
# =============================================================================
from pathlib import Path
import json

# Tự động quét tìm all_chunks.json trong các vị trí mount khả dĩ của Kaggle
POSSIBLE_INPUTS = [
    Path("/kaggle/input/ai-coding-tutor-rag-data/all_chunks.json"),
    Path("/kaggle/input/all_chunks.json"),
    Path("/kaggle/input/ai-coding-tutor/data/06_rag_dataset/chunks/all_chunks.json"),
    Path("./data/06_rag_dataset/chunks/all_chunks.json")  # Fallback
]

INPUT_CHUNK_FILE = None
for p in POSSIBLE_INPUTS:
    if p.exists():
        INPUT_CHUNK_FILE = p
        break

if not INPUT_CHUNK_FILE:
    raise FileNotFoundError(
        "❌ Không tìm thấy all_chunks.json! Hãy thêm Kaggle Dataset hoặc tải all_chunks.json lên /kaggle/input/."
    )

print(f"✅ Đã tìm thấy dữ liệu đầu vào: {INPUT_CHUNK_FILE} ({INPUT_CHUNK_FILE.stat().st_size:,} bytes)")

# Cấu hình thư mục làm việc và lưu trữ vector
WORKING_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = WORKING_DIR / "checkpoints"
EMBEDDINGS_DIR = WORKING_DIR / "embeddings"
REPORTS_DIR = WORKING_DIR / "reports"

for d in [CHECKPOINT_DIR, EMBEDDINGS_DIR, REPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Thư mục Checkpoint: {CHECKPOINT_DIR}")
print(f"✅ Thư mục Embeddings: {EMBEDDINGS_DIR}")
print(f"✅ Thư mục Báo cáo:   {REPORTS_DIR}")

In [ ]:
# =============================================================================
# 4. NHÚNG MÃ NGUỒN EMBEDDING FRAMEWORK TRONG NOTEBOOK
# =============================================================================
import re
import math
import time
import datetime
from enum import Enum
from typing import List, Dict, Tuple, Set, Optional

class Modality(str, Enum):
    TEXT = "text"
    CODE = "code"

class ModalityRouter:
    """Định tuyến Modality theo quy tắc trọng tâm ngữ nghĩa."""
    def __init__(self, code_ratio_threshold: float = 0.5):
        self.code_ratio_threshold = code_ratio_threshold
        self._code_pattern = re.compile(r"```(?:[a-zA-Z0-9_\-\.\+]+)?\s*([\s\S]*?)```", re.MULTILINE)

    def calculate_code_ratio(self, text: str) -> float:
        if not text:
            return 0.0
        matches = self._code_pattern.findall(text)
        if not matches:
            return 0.0
        return sum(len(m) for m in matches) / len(text)

    def classify_chunk(self, chunk: dict) -> Tuple[Modality, float]:
        metadata = chunk.get("metadata", {})
        content_type = metadata.get("content_type", "")
        has_code = metadata.get("has_code", False)
        text = chunk.get("text", "")

        if content_type == "code_example":
            return Modality.CODE, self.calculate_code_ratio(text)
        if has_code:
            ratio = self.calculate_code_ratio(text)
            if ratio >= self.code_ratio_threshold:
                return Modality.CODE, ratio
            return Modality.TEXT, ratio
        return Modality.TEXT, 0.0

    def route(self, chunks: List[dict]) -> Dict[Modality, List[dict]]:
        groups = {Modality.TEXT: [], Modality.CODE: []}
        for c in chunks:
            mod, _ = self.classify_chunk(c)
            groups[mod].append(c)
        return groups

class CheckpointManager:
    """Quản lý checkpoints bền vững trên Kaggle."""
    def __init__(self, base_dir: Path):
        self.base_dir = Path(base_dir)
        self.base_dir.mkdir(parents=True, exist_ok=True)

    def load_completed_ids(self, model_alias: str) -> Set[str]:
        mdir = self.base_dir / model_alias
        if not mdir.exists():
            return set()
        completed = set()
        for f in mdir.glob("batch_*.jsonl"):
            with open(f, "r", encoding="utf-8") as inf:
                for line in inf:
                    if line.strip():
                        rec = json.loads(line)
                        if rec.get("chunk_id"):
                            completed.add(rec["chunk_id"])
        return completed

    def save_batch(self, model_alias: str, batch_idx: int, records: List[dict]):
        mdir = self.base_dir / model_alias
        mdir.mkdir(parents=True, exist_ok=True)
        fpath = mdir / f"batch_{batch_idx:05d}.jsonl"
        tmp_path = mdir / f"batch_{batch_idx:05d}.jsonl.tmp"
        with open(tmp_path, "w", encoding="utf-8") as outf:
            for r in records:
                outf.write(json.dumps(r, ensure_ascii=False) + "\n")
        tmp_path.replace(fpath)

    def consolidate(self, model_alias: str, out_file: Path) -> int:
        mdir = self.base_dir / model_alias
        files = sorted(list(mdir.glob("batch_*.jsonl")))
        out_file.parent.mkdir(parents=True, exist_ok=True)
        count = 0
        with open(out_file, "w", encoding="utf-8") as outf:
            for f in files:
                with open(f, "r", encoding="utf-8") as inf:
                    for line in inf:
                        if line.strip():
                            outf.write(line)
                            count += 1
        return count

class VectorValidator:
    """Kiểm định chất lượng vector."""
    @staticmethod
    def validate(record: dict, expected_dim: int, seen_ids: Set[str]) -> Tuple[bool, List[str]]:
        errs = []
        cid = record.get("chunk_id")
        if not cid:
            errs.append("Missing chunk_id")
        elif cid in seen_ids:
            errs.append(f"Duplicate ID: {cid}")
        
        vec = record.get("vector")
        if not vec:
            errs.append("Vector is empty")
            return False, errs
        
        if len(vec) != expected_dim:
            errs.append(f"Dim mismatch: expected {expected_dim}, got {len(vec)}")
        
        if any(math.isnan(v) for v in vec):
            errs.append("Vector has NaN")
        if any(math.isinf(v) for v in vec):
            errs.append("Vector has Inf")
        return len(errs) == 0, errs

print("✅ Embedding Framework components loaded successfully.")

In [ ]:
# =============================================================================
# 5. NẠP DỮ LIỆU & ĐỐI SOÁT ĐỊNH TUYẾN MODALITY
# =============================================================================
with open(INPUT_CHUNK_FILE, "r", encoding="utf-8") as f:
    all_chunks = json.load(f)

router = ModalityRouter(code_ratio_threshold=0.5)
partitions = router.route(all_chunks)

text_chunks = partitions[Modality.TEXT]
code_chunks = partitions[Modality.CODE]

print(f"📊 TỔNG KẾT PHÂN BỔ MODALITY:")
print(f"  • Tổng số chunks: {len(all_chunks):,}")
print(f"  • Chunks TEXT   : {len(text_chunks):,} ({len(text_chunks)/len(all_chunks)*100:.2f}%)")
print(f"  • Chunks CODE   : {len(code_chunks):,} ({len(code_chunks)/len(all_chunks)*100:.2f}%)")

assert len(text_chunks) == 4287, f"Lỗi đếm TEXT: {len(text_chunks)} != 4287"
assert len(code_chunks) == 943, f"Lỗi đếm CODE: {len(code_chunks)} != 943"
print("🎉 Đối soát hoàn hảo 100%: 4,287 TEXT và 943 CODE!")

In [ ]:
# =============================================================================
# 6. PIPELINE A: CHẠY QWEN3-EMBEDDING-0.6B (CHO 4,287 TEXT CHUNKS)
# =============================================================================
import torch
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm

MODEL_ID_QWEN = "Qwen/Qwen3-Embedding-0.6B"
DIM_QWEN = 1024
BATCH_SIZE_QWEN = 32

print(f"=== BẮT ĐẦU NẠP MODEL: {MODEL_ID_QWEN} ===")
tokenizer_qwen = AutoTokenizer.from_pretrained(MODEL_ID_QWEN, padding_side="left")
model_qwen = AutoModel.from_pretrained(
    MODEL_ID_QWEN,
    torch_dtype=torch.float16,
    device_map="auto"
).eval()

def last_token_pool(last_hidden_states, attention_mask):
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

# Khởi tạo Checkpoint Manager
ckpt_mgr = CheckpointManager(CHECKPOINT_DIR)
completed_qwen_ids = ckpt_mgr.load_completed_ids("qwen3_text")
pending_qwen = [c for c in text_chunks if c["id"] not in completed_qwen_ids]
print(f"Qwen3 Chunks: Tổng {len(text_chunks)} | Đã xong: {len(completed_qwen_ids)} | Cần chạy: {len(pending_qwen)}")

# LƯU Ý GIAI ĐOẠN 3: Để thử nghiệm nhanh 100-200 chunks đầu, bỏ comment dòng dưới:
# pending_qwen = pending_qwen[:150]

num_batches = (len(pending_qwen) + BATCH_SIZE_QWEN - 1) // BATCH_SIZE_QWEN
seen_qwen_ids = set(completed_qwen_ids)

with torch.no_grad():
    for b_idx in tqdm(range(num_batches), desc="Embedding Qwen3"):
        batch_chunks = pending_qwen[b_idx * BATCH_SIZE_QWEN : (b_idx + 1) * BATCH_SIZE_QWEN]
        texts = [c["text"] for c in batch_chunks]
        
        inputs = tokenizer_qwen(texts, max_length=8192, padding=True, truncation=True, return_tensors="pt").to(model_qwen.device)
        outputs = model_qwen(**inputs)
        embeddings = last_token_pool(outputs.last_hidden_state, inputs["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1).cpu().to(torch.float32).tolist()
        
        valid_records = []
        for chunk, vec in zip(batch_chunks, embeddings):
            rec = {
                "chunk_id": chunk["id"],
                "source_file": chunk.get("metadata", {}).get("source_file", ""),
                "section_path": chunk.get("metadata", {}).get("section_path", ""),
                "modality": "text",
                "model_name": MODEL_ID_QWEN,
                "embedding_dimension": DIM_QWEN,
                "vector": vec,
                "token_count": chunk.get("metadata", {}).get("token_count", 0),
                "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat()
            }
            is_valid, errs = VectorValidator.validate(rec, DIM_QWEN, seen_qwen_ids)
            if is_valid:
                seen_qwen_ids.add(chunk["id"])
                valid_records.append(rec)
        
        ckpt_mgr.save_batch("qwen3_text", b_idx, valid_records)

# Hợp nhất vector
final_qwen_file = EMBEDDINGS_DIR / "qwen3_text_vectors.jsonl"
total_qwen = ckpt_mgr.consolidate("qwen3_text", final_qwen_file)
print(f"🎉 Hoàn tất Qwen3: Đã lưu {total_qwen:,} vectors tại {final_qwen_file}")

# GIẢI PHÓNG VRAM CHO MODEL KẾ TIẾP
del model_qwen
del tokenizer_qwen
torch.cuda.empty_cache()
print("🧹 Đã giải phóng VRAM của Qwen3!")

In [ ]:
# =============================================================================
# 7. PIPELINE B: CHẠY JINA-CODE (CHO 943 CODE CHUNKS)
# =============================================================================
from sentence_transformers import SentenceTransformer

MODEL_ID_JINA = "jinaai/jina-embeddings-v2-base-code"
DIM_JINA = 768
BATCH_SIZE_JINA = 32

print(f"=== BẮT ĐẦU NẠP MODEL: {MODEL_ID_JINA} ===")
model_jina = SentenceTransformer(MODEL_ID_JINA, trust_remote_code=True, device="cuda")

completed_jina_ids = ckpt_mgr.load_completed_ids("jina_code")
pending_jina = [c for c in code_chunks if c["id"] not in completed_jina_ids]
print(f"Jina Chunks: Tổng {len(code_chunks)} | Đã xong: {len(completed_jina_ids)} | Cần chạy: {len(pending_jina)}")

num_batches_jina = (len(pending_jina) + BATCH_SIZE_JINA - 1) // BATCH_SIZE_JINA
seen_jina_ids = set(completed_jina_ids)

for b_idx in tqdm(range(num_batches_jina), desc="Embedding Jina Code"):
    batch_chunks = pending_jina[b_idx * BATCH_SIZE_JINA : (b_idx + 1) * BATCH_SIZE_JINA]
    texts = [c["text"] for c in batch_chunks]
    
    embeddings = model_jina.encode(texts, batch_size=BATCH_SIZE_JINA, normalize_embeddings=True, show_progress_bar=False).tolist()
    
    valid_records = []
    for chunk, vec in zip(batch_chunks, embeddings):
        rec = {
            "chunk_id": chunk["id"],
            "source_file": chunk.get("metadata", {}).get("source_file", ""),
            "section_path": chunk.get("metadata", {}).get("section_path", ""),
            "modality": "code",
            "model_name": MODEL_ID_JINA,
            "embedding_dimension": DIM_JINA,
            "vector": vec,
            "token_count": chunk.get("metadata", {}).get("token_count", 0),
            "created_at": datetime.datetime.now(datetime.timezone.utc).isoformat()
        }
        is_valid, errs = VectorValidator.validate(rec, DIM_JINA, seen_jina_ids)
        if is_valid:
            seen_jina_ids.add(chunk["id"])
            valid_records.append(rec)
    
    ckpt_mgr.save_batch("jina_code", b_idx, valid_records)

final_jina_file = EMBEDDINGS_DIR / "jina_code_vectors.jsonl"
total_jina = ckpt_mgr.consolidate("jina_code", final_jina_file)
print(f"🎉 Hoàn tất Jina Code: Đã lưu {total_jina:,} vectors tại {final_jina_file}")

del model_jina
torch.cuda.empty_cache()
print("🧹 Đã giải phóng VRAM của Jina Code!")

In [ ]:
# =============================================================================
# 8. TỔNG KẾT, XUẤT BÁO CÁO & ĐÓNG GÓI OUTPUT ĐỂ TẢI HOẶC TẠO DATASET
# =============================================================================
import tarfile

report_data = {
    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "total_chunks": len(all_chunks),
    "qwen3_text_vectors": total_qwen,
    "jina_code_vectors": total_jina,
    "total_embedded": total_qwen + total_jina,
    "coverage_percentage": round((total_qwen + total_jina) / len(all_chunks) * 100, 2)
}

report_path = REPORTS_DIR / "kaggle_embedding_report.json"
with open(report_path, "w", encoding="utf-8") as rf:
    json.dump(report_data, rf, ensure_ascii=False, indent=2)

print("📊 BÁO CÁO NGHIỆM THU CUỐI CÙNG:")
print(json.dumps(report_data, indent=2))

# Đóng gói nén tar.gz tại /kaggle/working/ để người dùng tải 1 click
archive_path = WORKING_DIR / "ai_coding_tutor_embeddings.tar.gz"
with tarfile.open(archive_path, "w:gz") as tar:
    tar.add(EMBEDDINGS_DIR, arcname="embeddings")
    tar.add(REPORTS_DIR, arcname="reports")

print(f"\n📦 ĐÃ ĐÓNG GÓI THÀNH CÔNG: {archive_path} ({archive_path.stat().st_size:,} bytes)")
print("👉 Bạn có thể tải file này trực tiếp từ tab 'Output' của Kaggle Notebook!")